In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_orders = spark.table("workspace.final_project.silver_orders")
silver_items = spark.table("workspace.final_project.silver_items")
silver_suppliers = spark.table("workspace.final_project.silver_suppliers")
silver_incidents = spark.table("workspace.final_project.silver_incidents")

In [0]:
dim_suppliers = (
    silver_suppliers
    .select(
        "supplier_id",
        "supplier_name",
        "city",
        "country",
        "country_code"
    )
)

dim_suppliers.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_dim_suppliers"
)

In [0]:
fact_order_items = (
    silver_items
    .join(silver_orders, "order_id")
    .withColumn("order_month", F.trunc("order_date", "month"))
    .withColumn("spend", F.col("quantity") * F.col("unit_price"))
)

fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.final_project.gold_fact_order_items")

In [0]:
fact_incidents = (
    silver_incidents
    .join(
        silver_orders.select("order_id", "supplier_id", "order_date"),
        "order_id",
        "left"
    )
    .withColumn("incident_month", F.trunc("incident_date", "month"))
    .withColumn("incident_year", F.trunc("incident_date", "year"))
)

fact_incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_fact_incidents"
)

In [0]:
incidents_monthly = (
    fact_incidents
    .withColumn("year", F.year("incident_date"))
    .withColumn("month", F.month("incident_date"))
    .groupBy("supplier_id", "year", "month")
    .agg(F.count("*").alias("incident_count"))
)

incidents_monthly.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_fact_incidents_monthly"
)

In [0]:
from pyspark.sql import functions as F

# Calculer order_value et total_quantity depuis items en groupant par order_id
order_items_agg = (
    fact_order_items
    .groupBy("order_id", "product_category")
    .agg(
        F.sum("spend").alias("order_value"),
        F.sum("quantity").alias("total_quantity"),
        F.count("item_id").alias("total_items")
    )
)

# Calculer incident_count et les counts par severity depuis incidents en groupant par order_id
fact_incidents_agg = (
    fact_incidents
    .groupBy("order_id")
    .agg(
        F.count("incident_id").alias("incident_count"),
        F.sum(F.when(F.col("severity")=="Critical", 1).otherwise(0)).alias("nb_critical_incidents"),
        F.sum(F.when(F.col("severity")=="High", 1).otherwise(0)).alias("nb_high_incidents"),
        F.sum(F.when(F.col("severity")=="Medium", 1).otherwise(0)).alias("nb_medium_incidents"),
        F.sum(F.when(F.col("severity")=="Low", 1).otherwise(0)).alias("nb_low_incidents")
    )
)

# Joindre avec silver_order pour avoir dates et supplier_id
orders = (
    silver_orders
    .join(order_items_agg, on="order_id", how="left")
    .join(fact_incidents_agg, on="order_id", how="left")
    .fillna({
        "incident_count": 0,
        "nb_critical_incidents": 0,
        "nb_high_incidents": 0,
        "nb_medium_incidents": 0,
        "nb_low_incidents": 0
    })
    .withColumn(
        "on_time_delivery",
        F.when(
            F.col("delivery_date_actual") <= F.col("delivery_date_expected"),
            F.lit(1)
        ).otherwise(0)
    )
    .withColumn(
        "delay_days",
        F.datediff(F.col("delivery_date_actual"), F.col("delivery_date_expected"))
    )
        .withColumn(
        "delay_days",
        F.datediff(F.col("delivery_date_actual"), F.col("delivery_date_expected"))
    )
)

# Sélectionner les colonnes finales
orders = orders.select(
    "order_id",
    "supplier_id",
    "order_date",
    "product_category",
    "order_value",
    "total_quantity",
    "on_time_delivery",
    "delay_days",
    "incident_count",
    "nb_critical_incidents",
    "nb_high_incidents",
    "nb_medium_incidents",
    "nb_low_incidents"

)

# Rajouter .option("mergeSchema", "true") \ si problème de format
orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.final_project.gold_fact_orders")

In [0]:
group_by_orders = (orders.groupBy("product_category").agg(F.countDistinct("order_id").alias("nb_orders")))

display(group_by_orders)

In [0]:
from pyspark.sql import functions as F

agg_exprs = [
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("total_quantity").alias("total_quantity"),
    F.sum("order_value").alias("total_spend"),
    F.avg("order_value").alias("avg_order_value"),
    F.avg("on_time_delivery").alias("on_time_delivery_rate"),
    F.avg("delay_days").alias("avg_delay_days"),

    # Incident metrics
    F.sum("incident_count").alias("incident_count"),
    F.avg("incident_count").alias("incident_rate"),

    F.sum("nb_critical_incidents").alias("nb_critical_incidents"),
    F.sum("nb_high_incidents").alias("nb_high_incidents"),
    F.sum("nb_medium_incidents").alias("nb_medium_incidents"),
    F.sum("nb_low_incidents").alias("nb_low_incidents"),

    F.avg("nb_critical_incidents").alias("critical_incidents_rate"),
    F.avg("nb_high_incidents").alias("high_incidents_rate"),
    F.avg("nb_medium_incidents").alias("medium_incidents_rate"),
    F.avg("nb_low_incidents").alias("low_incidents_rate"),

    # Product category counts
    F.sum(F.when(F.col("product_category") == "electronics", 1).otherwise(0)).alias("nb_electronics"),
    F.sum(F.when(F.col("product_category") == "accessories", 1).otherwise(0)).alias("nb_accessories"),
    F.sum(F.when(F.col("product_category") == "industrial_tools", 1).otherwise(0)).alias("nb_industrial_tools"),
    F.sum(F.when(F.col("product_category") == "mechanical_parts", 1).otherwise(0)).alias("nb_mechanical_parts"),
    F.sum(F.when(F.col("product_category") == "medical_supply", 1).otherwise(0)).alias("nb_medical_supply"),
]

supplier_monthly = (
    orders
    .withColumn("month_date", F.trunc("order_date", "month"))
    .groupBy("supplier_id", "month_date")
    .agg(*agg_exprs)
    .withColumn("year", F.year("month_date"))
    .withColumn("month", F.month("month_date"))
)

date_bounds = supplier_monthly.agg(
    F.min("month_date").alias("min_date"),
    F.max("month_date").alias("max_date")
).first()

calendar = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{date_bounds["min_date"]}'),
        to_date('{date_bounds["max_date"]}'),
        interval 1 month
    )
) AS month_date
""")

window_3m = Window.partitionBy("supplier_id").orderBy("month_date").rowsBetween(-2, 0)

supplier_monthly_full = (
    supplier_monthly.select("supplier_id").distinct()
    .crossJoin(calendar)
    .join(supplier_monthly, ["supplier_id", "month_date"], "left")
    .fillna({
        "total_orders": 0,
        "total_quantity": 0,
        "total_spend": 0.0
    })
    .withColumn("year", F.year("month_date"))
    .withColumn("month", F.month("month_date"))
    .withColumn("rolling_orders_3m", F.sum("total_orders").over(window_3m))
    .withColumn("rolling_spend_3m", F.sum("total_spend").over(window_3m))
    .withColumn("rolling_incident_rate_3m", F.avg("incident_rate").over(window_3m))
    .withColumn("rolling_delivery_rate_3m", F.avg("on_time_delivery_rate").over(window_3m))
    .orderBy("supplier_id", "month_date")
)

In [0]:
supplier_monthly.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_monthly"
)

supplier_monthly_full.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_monthly_full"
)

In [0]:
from pyspark.sql import functions as F

# Agrégation annuelle par supplier
supplier_yearly = (
    orders
    .withColumn("year", F.year("order_date"))
    .groupBy("supplier_id", "year")
    .agg(*agg_exprs)
)

# Bornes des années
date_bounds = supplier_yearly.agg(
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year")
).first()

min_year = date_bounds["min_year"]
max_year = date_bounds["max_year"]

# Calendrier complet supplier x years
years = spark.range(min_year, max_year + 1).withColumnRenamed("id", "year")

supplier_yearly_full = (
    supplier_yearly.select("supplier_id").distinct()
    .crossJoin(years)
    .join(supplier_yearly, ["supplier_id", "year"], "left")
    .fillna({
        "total_orders": 0,
        "total_quantity": 0,
        "total_spend": 0.0
    })
    .orderBy("supplier_id", "year")
)

display(supplier_yearly_full)

In [0]:
# Nombre total d'incidents par fournisseur dans le temps
display(supplier_yearly)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
supplier_yearly.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_yearly"
)

supplier_yearly_full.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_yearly_full"
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------------------
# Agrégation annuelle par supplier et product_category
# -----------------------------------------
supplier_yearly_cat = (
    orders
    .withColumn("year", F.year("order_date"))
    .groupBy("supplier_id", "year", "product_category")
    .agg(*agg_exprs)
)

# -----------------------------------------
# Bornes des années
# -----------------------------------------
date_bounds = supplier_yearly_cat.agg(
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year")
).first()

min_year = date_bounds["min_year"]
max_year = date_bounds["max_year"]

# -----------------------------------------
# Liste des années
# -----------------------------------------
years = spark.range(min_year, max_year + 1).withColumnRenamed("id", "year")

# -----------------------------------------
# Liste des catégories distinctes
# -----------------------------------------
categories = supplier_yearly_cat.select("product_category").distinct()

# -----------------------------------------
# Calendrier complet supplier × year × category
# -----------------------------------------
supplier_yearly_cat_full = (
    supplier_yearly_cat.select("supplier_id").distinct()
    .crossJoin(years)
    .crossJoin(categories)
    .join(supplier_yearly_cat, ["supplier_id", "year", "product_category"], "left")
    .fillna({
        "total_orders": 0,
        "total_quantity": 0,
        "total_spend": 0.0
    })
    .orderBy("supplier_id", "year", "product_category")
)

display(supplier_yearly_cat_full)

In [0]:
supplier_yearly.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_yearly_cat"
)

supplier_yearly_full.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_performance_yearly_cat_full"
)

In [0]:
category_metrics = (
    fact_order_items
    .groupBy(
        "product_category",
        F.year("order_date").alias("year")
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("spend").alias("total_spend"),
        F.avg("unit_price").alias("avg_unit_price"),
        F.countDistinct("supplier_id").alias("supplier_count")
    )
)

category_metrics.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_category_metrics_yearly"
)

In [0]:
# Bar chart — total spend par catégorie et année
display(
    category_metrics.select("product_category", "year", "total_spend")
)

# Line chart — évolution du nombre de commandes par catégorie
display(
    category_metrics.select("product_category", "year", "total_orders")
)

# Bar chart ou pie chart — nombre de fournisseurs par catégorie
display(
    category_metrics.select("product_category", "year", "supplier_count")
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# -----------------------------------------
# Calcul des scores
# -----------------------------------------

# Delivery score (% commandes à temps)
supplier_scores = supplier_yearly.withColumn(
    "delivery_score",
    F.col("on_time_delivery_rate") * 100
)

# FIX IMPORTANT : incident_score borné entre 0 et 100
# incident_count peut être > total_orders
incident_rate = F.col("incident_count") / F.col("total_orders")

supplier_scores = supplier_scores.withColumn(
    "incident_rate_capped",
    F.when(incident_rate > 1, 1).otherwise(incident_rate)
)

supplier_scores = supplier_scores.withColumn(
    "incident_score",
    (1 - F.col("incident_rate_capped")) * 100
)

# Activity score pondéré par spend
max_spend = supplier_scores.agg(F.max("total_spend")).collect()[0][0]

supplier_scores = supplier_scores.withColumn(
    "activity_score",
    (F.col("total_spend") / F.lit(max_spend)) * 100
)

# Score global pondéré
supplier_scores = supplier_scores.withColumn(
    "supplier_score",
    0.5 * F.col("delivery_score") +
    0.3 * F.col("incident_score") +
    0.2 * F.col("activity_score")
)

# -----------------------------------------
# Classement des fournisseurs
# -----------------------------------------
window_rank = Window.orderBy(F.col("supplier_score").desc())

supplier_scores = supplier_scores.withColumn(
    "supplier_rank",
    F.rank().over(window_rank)
)

# -----------------------------------------
# 4️ Écriture dans Gold
# -----------------------------------------
supplier_scores.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    "workspace.final_project.gold_supplier_scores_yearly"
)

# -----------------------------------------
# 5️ Display / KPI 
# -----------------------------------------

# 5a — Classement top 20 fournisseurs
display(
    supplier_scores.select("supplier_rank", "supplier_id", "supplier_score")
    .orderBy(F.col("supplier_score").desc())
    .limit(20)
)

# 5b — Composition des scores pour top 10 fournisseurs (bar chart ou radar chart)
display(
    supplier_scores.select(
        "supplier_id", "delivery_score", "incident_score", "activity_score", "supplier_score"
    ).orderBy(F.col("supplier_score").desc()).limit(10)
)

# 5c — Distribution des scores (histogramme)
display(
    supplier_scores.select("supplier_score")
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.